# Engenharia de Features Multianual (2020-2022) - Datathon Passos Mágicos

## Introdução
Este notebook documenta o processo de criação de um dataset consolidado utilizando dados históricos de 2020, 2021 e 2022. Ao empilhar os dados de diferentes anos, aumentamos a robustez do modelo e permitimos que ele aprenda padrões temporais de desenvolvimento.

**Etapas:**
1. Extração de indicadores por ano (2020, 2021, 2022).
2. Empilhamento (Stacking) dos dados em um único dataframe.
3. Padronização e Tratamento de Nulos.
4. Recriação de Features e Definição do Target.
5. Exportação do Dataset Consolidado.

## 1. Extração e Empilhamento de Dados

Identificamos as colunas correspondentes a cada ano e as consolidamos em um formato 'longo'.

In [ ]:
import pandas as pd
import numpy as np
import os

# Carregamento do CSV original
df_raw = pd.read_csv('../data/raw/PEDE_PASSOS_DATASET_FIAP.csv', sep=';')

# Função para extrair e padronizar dados de um ano específico
def extract_year_data(df, year):
    cols = [f'INDE_{year}', f'IDA_{year}', f'IEG_{year}', f'IAA_{year}', f'IPS_{year}', f'IPP_{year}', f'IAN_{year}', f'IPV_{year}']
    # Verificar se as colunas existem no dataframe
    existing_cols = [c for c in cols if c in df.columns]
    df_year = df[existing_cols].copy()
    # Padronizar nomes (remover o sufixo do ano)
    df_year.columns = [c.split('_')[0] for c in df_year.columns]
    df_year['year'] = year
    return df_year

# Extrair dados de 2020, 2021 e 2022
df_2020 = extract_year_data(df_raw, 2020)
df_2021 = extract_year_data(df_raw, 2021)
df_2022 = extract_year_data(df_raw, 2022)

# Empilhar (Stacking)
df_stacked = pd.concat([df_2020, df_2021, df_2022], ignore_index=True)

# Conversão Numérica
def to_numeric(col_data):
    return pd.to_numeric(col_data.astype(str).str.replace(',', '.'), errors='coerce')

indicadores = ['INDE', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IAN', 'IPV']
for col in indicadores:
    if col in df_stacked.columns:
        df_stacked[col] = to_numeric(df_stacked[col])

print(f'Dataset empilhado com {df_stacked.shape[0]} registros.')
df_stacked.head()

## 2. Tratamento de Nulos e Engenharia de Features

Limpamos os dados e recriamos as variáveis preditivas.

In [ ]:
# 1. Tratamento de Nulos (Mediana por Ano)
for col in indicadores:
    df_stacked[col] = df_stacked.groupby('year')[col].transform(lambda x: x.fillna(x.median()))

# 2. Engenharia de Features
df_stacked['perception_gap'] = df_stacked['IAA'] - df_stacked['IDA']
df_stacked['behavioral_score'] = (df_stacked['IEG'] + df_stacked['IPS']) / 2

# Performance Relativa (IDA - média do IDA do respectivo ano)
df_stacked['relative_performance'] = df_stacked.groupby('year')['IDA'].transform(lambda x: x - x.mean())

# 3. Definição do Target
df_stacked['risk_gap'] = (df_stacked['IAN'] < 6).astype(int)

df_stacked.describe()

## 3. Exportação do Dataset Consolidado

Salvando o dataset para o treinamento do modelo.

In [ ]:
output_path = '../data/processed/processed_data.csv'
df_stacked.to_csv(output_path, index=False)
print(f'Dataset multianual exportado para: {output_path}')